# Silver Layer Transformation

This notebook implements the **Silver layer** of the Medallion Architecture.

It reads raw Bronze Delta tables, applies cleaning, type normalization, enrichment, and business logic
to produce validated, analytics-ready datasets. Transformations include:
- **Costs:** FOCUS column normalization, date parsing, Fabric-only filtering, windowed cost analytics
- **Metrics:** Pivot metric names into columns, resample to hourly/daily granularity, null handling
- **Logs:** Structured field parsing, severity extraction, source categorization
- **Resources:** Tag flattening, enrichment via cost data joins

In [ ]:
from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import (
    col, to_date, to_timestamp, when, lit, coalesce, sum as _sum,
    count, avg, max as _max, min as _min, first, round as _round,
    from_json, explode, split, trim, lower, upper, regexp_extract,
    window, date_trunc, row_number, dense_rank, current_timestamp,
    isnull, isnan
)
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, MapType
)

spark = SparkSession.builder.getOrCreate()

## 1. Read Bronze Delta Tables

Load all four Bronze tables produced by the ingestion notebook.

In [ ]:
df_bronze_costs = spark.read.format("delta").load("Tables/bronze_costs")
df_bronze_metrics = spark.read.format("delta").load("Tables/bronze_metrics")
df_bronze_logs = spark.read.format("delta").load("Tables/bronze_logs")
df_bronze_metadata = spark.read.format("delta").load("Tables/bronze_resource_metadata")

print(f"Bronze costs:    {df_bronze_costs.count()} rows")
print(f"Bronze metrics:  {df_bronze_metrics.count()} rows")
print(f"Bronze logs:     {df_bronze_logs.count()} rows")
print(f"Bronze metadata: {df_bronze_metadata.count()} rows")

## 2. Transform Costs

Normalize FOCUS cost columns, parse date fields, filter to Fabric-related services only,
and add windowed analytics (cost per day, running total).

In [ ]:
df_costs_normalized = (
    df_bronze_costs
    .select(
        col("BillingAccountId"),
        col("BillingAccountName"),
        col("SubscriptionId"),
        col("SubscriptionName"),
        col("ResourceId"),
        col("ResourceName"),
        col("ResourceType"),
        col("ServiceName"),
        col("ServiceCategory"),
        col("RegionId"),
        col("RegionName"),
        col("ChargeType"),
        col("ChargeFrequency"),
        col("PricingUnit"),
        col("UsageUnit"),
        col("BilledCost").cast("double").alias("billed_cost"),
        col("EffectiveCost").cast("double").alias("effective_cost"),
        col("PricingQuantity").cast("double").alias("pricing_quantity"),
        col("UsageQuantity").cast("double").alias("usage_quantity"),
        col("ListUnitPrice").cast("double").alias("list_unit_price"),
        col("ContractedUnitPrice").cast("double").alias("contracted_unit_price"),
        to_date(col("ChargePeriodStart")).alias("charge_period_start"),
        to_date(col("ChargePeriodEnd")).alias("charge_period_end"),
        to_timestamp(col("BillingPeriodStart")).alias("billing_period_start"),
        to_timestamp(col("BillingPeriodEnd")).alias("billing_period_end"),
        col("Tags"),
        col("_ingestion_timestamp"),
    )
    .withColumn(
        "billed_cost",
        coalesce(col("billed_cost"), lit(0.0))
    )
    .withColumn(
        "effective_cost",
        coalesce(col("effective_cost"), lit(0.0))
    )
)

print(f"Normalized cost records: {df_costs_normalized.count()}")

In [ ]:
df_costs_fabric = df_costs_normalized.filter(
    col("ServiceCategory").contains("Fabric")
    | col("ServiceName").contains("Microsoft.Fabric")
)

daily_window = Window.partitionBy("ResourceId").orderBy("charge_period_start")
cumulative_window = (
    Window.partitionBy("ResourceId", "SubscriptionId")
    .orderBy("charge_period_start")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df_silver_costs = (
    df_costs_fabric
    .withColumn(
        "cost_per_day",
        _sum("effective_cost").over(
            Window.partitionBy("ResourceId", "charge_period_start")
        )
    )
    .withColumn(
        "running_total",
        _sum("effective_cost").over(cumulative_window)
    )
    .withColumn("_silver_timestamp", current_timestamp())
)

print(f"Fabric-filtered cost records: {df_silver_costs.count()}")
df_silver_costs.select(
    "ResourceName", "charge_period_start", "effective_cost",
    "cost_per_day", "running_total"
).show(5, truncate=False)

## 3. Transform Metrics

Pivot metric names into columns, resample to hourly and daily granularity,
and fill null values with zero for numeric aggregations.

In [ ]:
df_metrics_parsed = (
    df_bronze_metrics
    .withColumn("event_timestamp", to_timestamp(col("TimeGenerated")))
    .withColumn("metric_name", trim(col("MetricName")))
    .withColumn("metric_value", col("MetricValue").cast("double"))
    .withColumn("resource_id", col("ResourceId"))
    .withColumn("metric_value", coalesce(col("metric_value"), lit(0.0)))
)

df_metrics_pivoted = (
    df_metrics_parsed
    .groupBy(
        "resource_id",
        window("event_timestamp", "1 hour").alias("time_window")
    )
    .pivot("metric_name")
    .agg(avg("metric_value"))
    .select(
        col("resource_id"),
        col("time_window.start").alias("window_start"),
        col("time_window.end").alias("window_end"),
        col("*")
    )
    .drop("time_window", "resource_id")
)

pivot_cols = [c for c in df_metrics_pivoted.columns if c not in
              ["resource_id", "window_start", "window_end"]]
df_metrics_hourly = df_metrics_pivoted.fillna(0.0, subset=pivot_cols)

print(f"Hourly metric records: {df_metrics_hourly.count()}")
df_metrics_hourly.printSchema()

In [ ]:
df_metrics_daily = (
    df_metrics_parsed
    .withColumn("metric_date", to_date(col("event_timestamp")))
    .groupBy("resource_id", "metric_date", "metric_name")
    .agg(
        avg("metric_value").alias("avg_value"),
        _max("metric_value").alias("max_value"),
        _min("metric_value").alias("min_value"),
        count("metric_value").alias("sample_count")
    )
    .fillna(0.0, subset=["avg_value", "max_value", "min_value"])
)

df_silver_metrics = df_metrics_hourly.withColumn("_silver_timestamp", current_timestamp())

print(f"Daily metric summary records: {df_metrics_daily.count()}")

## 4. Transform Logs

Parse structured log fields, extract severity levels, and categorize log entries
by their source system.

In [ ]:
log_properties_schema = StructType([
    StructField("statusCode", StringType(), True),
    StructField("statusMessage", StringType(), True),
    StructField("durationMs", DoubleType(), True),
    StructField("correlationId", StringType(), True),
])

df_silver_logs = (
    df_bronze_logs
    .withColumn("event_timestamp", to_timestamp(col("TimeGenerated")))
    .withColumn("event_date", to_date(col("TimeGenerated")))
    .withColumn("resource_id", col("ResourceId"))
    .withColumn("operation_name", trim(col("OperationName")))
    .withColumn("category", trim(col("Category")))
    .withColumn(
        "severity",
        when(lower(col("Level")) == "critical", lit("CRITICAL"))
        .when(lower(col("Level")) == "error", lit("ERROR"))
        .when(lower(col("Level")) == "warning", lit("WARNING"))
        .when(lower(col("Level")) == "informational", lit("INFO"))
        .when(lower(col("Level")) == "verbose", lit("DEBUG"))
        .otherwise(lit("UNKNOWN"))
    )
    .withColumn(
        "source_system",
        when(col("category").contains("Audit"), lit("AUDIT"))
        .when(col("category").contains("Administrative"), lit("ADMIN"))
        .when(col("category").contains("Security"), lit("SECURITY"))
        .when(col("category").contains("Diagnostic"), lit("DIAGNOSTIC"))
        .when(col("category").contains("Pipeline"), lit("PIPELINE"))
        .when(col("category").contains("Spark"), lit("SPARK"))
        .otherwise(lit("OTHER"))
    )
    .withColumn(
        "parsed_properties",
        from_json(col("Properties"), log_properties_schema)
    )
    .withColumn("status_code", col("parsed_properties.statusCode"))
    .withColumn("duration_ms", col("parsed_properties.durationMs"))
    .withColumn("correlation_id", col("parsed_properties.correlationId"))
    .drop("parsed_properties")
    .withColumn("_silver_timestamp", current_timestamp())
)

print(f"Silver log records: {df_silver_logs.count()}")
df_silver_logs.groupBy("severity", "source_system").count().show(truncate=False)

## 5. Transform Resource Metadata

Flatten the tags JSON column into individual key-value rows, then join with
cost data to produce an enriched resource dimension table.

In [ ]:
tags_schema = MapType(StringType(), StringType())

df_resources_parsed = (
    df_bronze_metadata
    .withColumn("tags_map", from_json(col("tags"), tags_schema))
    .withColumn("resource_id", lower(col("id")))
    .withColumn("resource_name", col("name"))
    .withColumn("resource_type", col("type"))
    .withColumn("resource_group", col("resourceGroup"))
    .withColumn("subscription_id", col("subscriptionId"))
    .withColumn("location", col("location"))
    .withColumn("provisioning_state", col("properties.provisioningState"))
)

df_tags_exploded = (
    df_resources_parsed
    .select(
        "resource_id",
        explode("tags_map").alias("tag_key", "tag_value")
    )
)

df_tags_wide = (
    df_tags_exploded
    .groupBy("resource_id")
    .pivot("tag_key")
    .agg(first("tag_value"))
)

df_resources_with_tags = df_resources_parsed.join(
    df_tags_wide, on="resource_id", how="left"
).drop("tags_map")

cost_summary = (
    df_silver_costs
    .withColumn("resource_id_lower", lower(col("ResourceId")))
    .groupBy("resource_id_lower")
    .agg(
        _sum("effective_cost").alias("total_effective_cost"),
        _sum("billed_cost").alias("total_billed_cost"),
        _min("charge_period_start").alias("first_charge_date"),
        _max("charge_period_start").alias("last_charge_date"),
        count("*").alias("charge_line_count")
    )
)

df_silver_resources = (
    df_resources_with_tags
    .join(cost_summary, df_resources_with_tags.resource_id == cost_summary.resource_id_lower, "left")
    .drop("resource_id_lower")
    .withColumn("total_effective_cost", coalesce(col("total_effective_cost"), lit(0.0)))
    .withColumn("total_billed_cost", coalesce(col("total_billed_cost"), lit(0.0)))
    .withColumn("_silver_timestamp", current_timestamp())
)

print(f"Silver resource records: {df_silver_resources.count()}")
print(f"Tags extracted: {df_tags_exploded.count()} tag entries")

## 6. Data Quality Assertions

Verify that key columns contain no nulls after transformation. These assertions
will halt the notebook if a critical data quality issue is detected.

In [ ]:
def assert_no_nulls(df, column_name, table_name):
    """Assert that a column has no null values."""
    null_count = df.filter(isnull(col(column_name))).count()
    assert null_count == 0, (
        f"Data quality failure: {table_name}.{column_name} "
        f"has {null_count} null values"
    )
    print(f"  [PASS] {table_name}.{column_name}: no nulls")


def assert_positive_count(df, table_name):
    """Assert that a DataFrame has at least one row."""
    row_count = df.count()
    assert row_count > 0, (
        f"Data quality failure: {table_name} has zero rows"
    )
    print(f"  [PASS] {table_name}: {row_count} rows")


print("--- Silver Costs ---")
assert_positive_count(df_silver_costs, "silver_costs")
assert_no_nulls(df_silver_costs, "effective_cost", "silver_costs")
assert_no_nulls(df_silver_costs, "billed_cost", "silver_costs")
assert_no_nulls(df_silver_costs, "charge_period_start", "silver_costs")

print("\n--- Silver Metrics ---")
assert_positive_count(df_silver_metrics, "silver_metrics")
assert_no_nulls(df_silver_metrics, "window_start", "silver_metrics")
assert_no_nulls(df_silver_metrics, "window_end", "silver_metrics")

print("\n--- Silver Logs ---")
assert_positive_count(df_silver_logs, "silver_logs")
assert_no_nulls(df_silver_logs, "event_timestamp", "silver_logs")
assert_no_nulls(df_silver_logs, "severity", "silver_logs")
assert_no_nulls(df_silver_logs, "source_system", "silver_logs")

print("\n--- Silver Resources ---")
assert_positive_count(df_silver_resources, "silver_resources")
assert_no_nulls(df_silver_resources, "resource_id", "silver_resources")
assert_no_nulls(df_silver_resources, "resource_type", "silver_resources")

print("\nAll data quality assertions passed.")

## 7. Write Silver Delta Tables

Persist all transformed datasets as managed Delta tables in the Lakehouse.

In [ ]:
silver_tables = {
    "Tables/silver_costs": df_silver_costs,
    "Tables/silver_metrics": df_silver_metrics,
    "Tables/silver_logs": df_silver_logs,
    "Tables/silver_resources": df_silver_resources,
}

for table_path, df in silver_tables.items():
    table_name = table_path.split("/")[-1]
    print(f"Writing {table_name} to {table_path}...")
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(table_path)
    )
    written_count = spark.read.format("delta").load(table_path).count()
    print(f"  {table_name}: {written_count} rows written successfully.")

print("\nSilver layer transformation complete.")